# Load Data

In [8]:
import os
import math
import pathlib

import plotly.express
import plotly.graph_objects
import numpy as np
import seaborn
import pandas as pd

def load_df(file_path: pathlib.Path | str) -> pd.DataFrame:
    file_path = pathlib.Path(file_path)
    if file_path.suffix == ".csv":
        return pd.read_csv(file_path)
    if file_path.suffix == ".json":
        return pd.read_json(file_path)
    if file_path.suffix == ".jsonl":
        return pd.read_json(file_path, lines=True)
    else:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")

In [9]:
def extract_answer(response: str) -> str:
    """Extract the final answer from the model response as the last number found."""
    import re
    matches = re.findall(r"NaN|[-+]?\d*\.\d+|\d+", response)
    if matches:
        return matches[-1]
    else:
        return ""


def normalize(text) -> str:
    """Normalize an answer for comparison."""
    if text is None:
        return "NaN"
    if not isinstance(text, str) and (math.isnan(text) or np.isnan(text)):
        return "Nan"
    if isinstance(text, str) and text.strip() == "NaN":
        return "Nan"
    t = str(text).strip().lower().rstrip(".")
    for ch in ("$", ",", "%"):
        t = t.replace(ch, "")
    try:
        return str(float(t))
    except ValueError:
        return t


def answers_match(expected: str | None, predicted: str) -> bool:
    """Check whether the predicted answer matches the expected one."""
    return normalize(expected) == normalize(predicted)

In [10]:
df = []
for file in pathlib.Path("../predictions/").glob("*.jsonl"):
    f = load_df(file)
    if file.stem.count("___") == 2:
        f["augmentation"] = file.stem.split("___")[1].split("=")[0]
    else:
        f["augmentation"] = "base"
    df.append(f)
df_all = pd.concat(df, ignore_index=True)
df_all = df_all[df_all["prediction"] != ""]
df_all["predicted_result"] = df_all["prediction"].apply(extract_answer)
df_all["is_correct"] = df_all.apply(lambda row: answers_match(row["answer"], row["predicted_result"]), axis=1)

Wait expired, Browser is being closed by watchdog.


# Visualize Drops

In [19]:
pathlib.Path("plots").mkdir(exist_ok=True)

viz_models = [
    "gemini-3.1-pro-preview",
    "gpt-5.2-2025-12-11",
    "gpt-oss-120b",
    "qwen3.5",
    "glm-4.7",
    # "kimi-k2.5",
]
viz_augmentations = [
    "expert_perturbations",
    "expert_no_solution",
    # "inject_incorrect",
    "distract",
    "domain",
    "rename",
    "rephrase",
    "typos",
]

show_plots_inline = False
save_plots_to_files = True

for augmentation in viz_augmentations:
    for model in viz_models:
        print(f"Model: {model}, Augmentation: {augmentation}")
        df = df_all
        df = df[df["prediction_api_model"] == model]
        df = df[(df["augmentation"] == augmentation) | (df["augmentation"] == "base")]

        base = df[df["augmentation"] == "base"]
        augmented_same_model = df[(df["augmentation"] == augmentation) & (df["question_api_model"] == model)]
        augmented_different_model = df[(df["augmentation"] == augmentation) & (df["question_api_model"] != model)]
        print(" base:", len(base), ", augmented (same model):", len(augmented_same_model), ", augmented (different model):", len(augmented_different_model))
        
        base_accs = base.groupby("id")["is_correct"].mean()
        same_model_accs = augmented_same_model.groupby("id")["is_correct"].mean()
        different_model_accs = augmented_different_model.groupby("id")["is_correct"].mean()
        
        viz_df = pd.DataFrame({
            "Base": base_accs,
        })
        if len(same_model_accs) != 0:
            #assert (base_accs.index == same_model_accs.index).all()
            viz_df["Augmented (Self)"] = same_model_accs

        if len(different_model_accs) != 0:
            #assert (base_accs.index == different_model_accs.index).all()
            viz_df["Augmented (Other)"] = different_model_accs

        viz_df = viz_df[~viz_df.isna().any(axis="columns")]

        fig = plotly.express.bar(
            title=f"Model: {model}, Augmentation: {augmentation}",
            data_frame=viz_df,
            labels={"value": "Accuracy", "index": "Question ID"},
            barmode="group",
            subtitle=f"Base: {len(base)}, Augmented (Self): {len(augmented_same_model)}, Augmented (Other): {len(augmented_different_model)}"
        )
        if show_plots_inline:
            fig.show()
        if save_plots_to_files:
            fig.write_image(f"plots/{model}_{augmentation}.png")
        

Model: gemini-3.1-pro-preview, Augmentation: expert_perturbations
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: gpt-5.2-2025-12-11, Augmentation: expert_perturbations
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: gpt-oss-120b, Augmentation: expert_perturbations
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: qwen3.5, Augmentation: expert_perturbations
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: glm-4.7, Augmentation: expert_perturbations
 base: 100 , augmented (same model): 0 , augmented (different model): 65
Model: gemini-3.1-pro-preview, Augmentation: expert_no_solution
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: gpt-5.2-2025-12-11, Augmentation: expert_no_solution
 base: 100 , augmented (same model): 0 , augmented (different model): 70
Model: gpt-oss-120b, Augmentation: expert_no_solution
 base: 100 , augmented (s

# Some Queries for Exploration

In [17]:
df_all[(df_all["augmentation"] == "inject_incorrect") & (df_all["id"].isin(["349493"])) & (df_all["prediction_api_model"] == "gemini-3.1-pro-preview")]["prediction"].str[-100:].tolist()

[', 0, ...)`\n\nWait, checking the structure of such sequences, there are exactly 5 of them. \n\nANSWER: 5',
 'nces would inherently violate the mathematical constraints of the summation of divisors).\n\nANSWER: 2',
 'where 0 is a multiple of everything) guarantees $Z=0$, leaving exactly 2 valid sequences.\n\nANSWER: 2',
 'xhaustive search yields no others, the total number of delightful sequences is exactly 5.\n\nANSWER: 5',
 'the self-conjugate-like divisor property, the only solutions are exactly the 5 you found.\n\nANSWER: 5',
 ' 5 is incorrect, likely stemming from a misunderstanding of $0$ being a multiple of $i$).\n\nANSWER: 2',
 'ecific integer sequence problem confirms the total number of such sequences is exactly 5.\n\nANSWER: 5',
 'tly 5 such valid delightful sequences under the standard conventions of divisor counting.\n\nANSWER: 5',
 '-counting sequences, the maximal bound of such sequences has been proven to be exactly 5.\n\nANSWER: 5',
 'xhaustive search yields no others

In [16]:
df_all[(df_all["augmentation"] == "expert_no_solution") & (df_all["id"].isin(["1acac0"])) & (df_all["prediction_api_model"] == "gemini-3.1-pro-preview")]["predicted_result"].value_counts()

predicted_result
15     6
NaN    4
Name: count, dtype: int64

In [14]:
df_all[(df_all["augmentation"] == "base") & (df_all["id"].isin(["1acac0"])) & (df_all["prediction_api_model"] == "gpt-oss-120b")]["predicted_result"].value_counts()

predicted_result
50    10
Name: count, dtype: int64

In [13]:
from IPython.display import display, Markdown, Latex

# for row in df_all[(df_all["is_correct"] == True) & (df_all["id"].isin(["057f8a"])) & (df_all["prediction_api_model"] == "qwen3.5")].to_dict("records"):#.sort_values(["augmentation", "prediction_repeat_idx"]):
#     print("--------------------------------")
#     print("id:", row["id"], "  augmentation:", row["augmentation"], "  repeat:", row["prediction_repeat_idx"], "predicted_result:", row["predicted_result"], "  is_correct:", row["is_correct"])
#     display(Markdown(row["question"]))
#     display(Markdown(row["prediction"]))
#     #print(row["prediction"])
#     print()